In [ ]:
import os
from pathlib import Path

# --- EDIT THESE ---
IMAGES_ROOT = Path("/content/drive/MyDrive/temp")
OUTPUT_MODELS_DIR = Path("/content/drive/MyDrive/Deep Learning Project/MODELS")
IMAGES_PER_CLASS = 200     # <-- change this to use more/less images per class
TRAIN_VAL_SPLIT = 0.8      # fraction for training
RANDOM_SEED = 42
BATCH_SIZE = 32
NUM_EPOCHS = 8
IMG_SIZE = 224             # input resolution for ResNet
LEARNING_RATE = 1e-4
NUM_WORKERS = 4            # DataLoader workers in Colab keep small (0-4)
# -----------------------

# Safety checks / create dirs
assert IMAGES_ROOT.exists(), f"Images folder not found: {IMAGES_ROOT}"
OUTPUT_MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Images root:", IMAGES_ROOT)
print("Output models dir:", OUTPUT_MODELS_DIR)
print("Images per class:", IMAGES_PER_CLASS)


Images root: /content/drive/MyDrive/temp
Output models dir: /content/drive/MyDrive/Deep Learning Project/MODELS
Images per class: 200


In [ ]:
import random
import shutil
from pathlib import Path
from tqdm import tqdm

random.seed(RANDOM_SEED)

TMP_ROOT = Path("/tmp/food_subset")
TRAIN_DIR = TMP_ROOT / "train"
VAL_DIR = TMP_ROOT / "val"

# clear old tmp if exists (be careful)
if TMP_ROOT.exists():
    print("Removing existing tmp subset:", TMP_ROOT)
    shutil.rmtree(TMP_ROOT)

TRAIN_DIR.mkdir(parents=True, exist_ok=True)
VAL_DIR.mkdir(parents=True, exist_ok=True)

# assume class subfolders inside IMAGES_ROOT (food-101 convention)
classes = [p for p in IMAGES_ROOT.iterdir() if p.is_dir()]
classes = sorted(classes)
print(f"Found {len(classes)} classes. Sampling up to {IMAGES_PER_CLASS} images per class...")

for cls in tqdm(classes):
    imgs = list(cls.glob("*"))
    if len(imgs) == 0:
        continue
    random.shuffle(imgs)
    use = imgs[:IMAGES_PER_CLASS]  # up to
    # split per-class
    split_idx = int(len(use) * TRAIN_VAL_SPLIT)
    train_imgs = use[:split_idx]
    val_imgs = use[split_idx:]

    # create class dirs in train and val
    (TRAIN_DIR / cls.name).mkdir(parents=True, exist_ok=True)
    (VAL_DIR / cls.name).mkdir(parents=True, exist_ok=True)

    # copy
    for p in train_imgs:
        shutil.copy(p, TRAIN_DIR / cls.name / p.name)
    for p in val_imgs:
        shutil.copy(p, VAL_DIR / cls.name / p.name)

print("Subset created at:", TMP_ROOT)


Found 15 classes. Sampling up to 200 images per class...


100%|██████████| 15/15 [05:52<00:00, 23.50s/it]

Subset created at: /tmp/food_subset


In [ ]:
import torch
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.1)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
val_dataset   = datasets.ImageFolder(VAL_DIR, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print("Num train images:", len(train_dataset))
print("Num val images:", len(val_dataset))
print("Num classes:", len(train_dataset.classes))


Device: cpu
Num train images: 2400
Num val images: 600
Num classes: 15


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [ ]:
import torch.nn as nn
from torchvision import models
import torch.optim as optim

num_classes = len(train_dataset.classes)

# Load pretrained ResNet50 and replace head
model = models.resnet50(pretrained=True)
# Replace the final fully connected layer
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
# optional scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

print(model)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 138MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [ ]:
import time
from copy import deepcopy

best_model_wts = deepcopy(model.state_dict())
best_acc = 0.0
save_path = OUTPUT_MODELS_DIR / "best_resnet50_food_subset.pth"

for epoch in range(1, NUM_EPOCHS + 1):
    since = time.time()
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
    print("-" * 30)

    # --- Train ---
    model.train()
    running_loss = 0.0
    running_corrects = 0
    total_train = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        running_corrects += torch.sum(preds == labels.data).item()
        total_train += labels.size(0)

    epoch_loss = running_loss / total_train
    epoch_acc = running_corrects / total_train
    print(f"Train loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

    # --- Validate ---
    model.eval()
    val_loss = 0.0
    val_corrects = 0
    total_val = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            val_corrects += torch.sum(preds == labels.data).item()
            total_val += labels.size(0)

    val_loss = val_loss / total_val
    val_acc = val_corrects / total_val
    print(f"Val   loss: {val_loss:.4f} Acc: {val_acc:.4f}")

    # Step scheduler
    scheduler.step()

    # Save best
    if val_acc > best_acc:
        best_acc = val_acc
        best_model_wts = deepcopy(model.state_dict())
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'classes': train_dataset.classes
        }, save_path)
        print(f"Saved new best model to: {save_path}")

    print(f"Epoch time: {time.time() - since:.0f}s  Best val acc: {best_acc:.4f}")

# load best weights into model (optional)
model.load_state_dict(best_model_wts)
print("Training complete. Best val acc:", best_acc)



Epoch 1/8
------------------------------


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Train loss: 1.5857 Acc: 0.5508
Val   loss: 0.7624 Acc: 0.7633
Saved new best model to: /content/drive/MyDrive/Deep Learning Project/MODELS/best_resnet50_food_subset.pth
Epoch time: 2049s  Best val acc: 0.7633

Epoch 2/8
------------------------------
Train loss: 0.8783 Acc: 0.7362
Val   loss: 0.6781 Acc: 0.7900
Saved new best model to: /content/drive/MyDrive/Deep Learning Project/MODELS/best_resnet50_food_subset.pth
Epoch time: 2035s  Best val acc: 0.7900

Epoch 3/8
------------------------------
Train loss: 0.6547 Acc: 0.8013
Val   loss: 0.6013 Acc: 0.8250
Saved new best model to: /content/drive/MyDrive/Deep Learning Project/MODELS/best_resnet50_food_subset.pth
Epoch time: 2034s  Best val acc: 0.8250

Epoch 4/8
------------------------------
Train loss: 0.5067 Acc: 0.8508
Val   loss: 0.5186 Acc: 0.8550
Saved new best model to: /content/drive/MyDrive/Deep Learning Project/MODELS/best_resnet50_food_subset.pth
Epoch time: 2043s  Best val acc: 0.8550

Epoch 5/8
---------------------------

In [ ]:
import torch
from torchvision import models
import torch.nn as nn

save_path = "/content/drive/MyDrive/Deep Learning Project/MODELS/best_resnet50_food_subset.pth"
checkpoint = torch.load(save_path, map_location=device)
classes = checkpoint.get('classes')
num_classes = len(classes)

model = models.resnet50(pretrained=False)   # pretrained not necessary when loading weights
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print("Loaded model with", num_classes, "classes")
print("Classes (first 10):", classes[:10])


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Loaded model with 15 classes
Classes (first 10): ['apple pie', 'baby back ribs', 'baklava', 'beef carpaccio', 'donuts', 'french fries', 'french toast', 'fried rice', 'hamburger', 'ice cream']


In [ ]:
from PIL import Image
from torchvision import transforms
import torch
import numpy as np

def predict_image(img_path, model, classes, device, img_size=IMG_SIZE):
    preprocess = transforms.Compose([
        transforms.Resize(int(img_size * 1.1)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])
    img = Image.open(img_path).convert("RGB")
    x = preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(x)
        probs = torch.nn.functional.softmax(out, dim=1)
        conf, idx = torch.max(probs, 1)
    return classes[idx.item()], conf.item()

# Example usage:
# cls, confidence = predict_image('/content/some_image.jpg', model, classes, device)
# print(cls, confidence)


In [ ]:
# Example usage:
cls, confidence = predict_image('/content/drive/MyDrive/test images/Semi-Instant-Pancakes_Lynne_resized.jpg', model, classes, device)
print(cls, confidence)

pancakes 0.9468815326690674
